## **PROJET STEAM — Notebook 01 : Data Exploration & Cleaning**
**Objectif** : Charger le dataset JSON depuis S3, explorer sa structure, nettoyer les données et préparer les DataFrames pour l'analyse.


In [0]:
# Import package 
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, explode, split, trim, regexp_replace,
    regexp_extract, when, lit, coalesce,
    try_to_date, to_date, year, month,
    array_size,
    count, avg, sum as spark_sum, max as spark_max,
    stack
)
from pyspark.sql.types import IntegerType, FloatType, DoubleType, LongType
 
spark = SparkSession.builder.appName("steam-project").getOrCreate()

In [0]:
# 1. Chargement des données depuis S3
df_raw = (spark.read 
            .option("multiLine", "true") 
            .json("s3://full-stack-bigdata-datasets/Big_Data/Project_Steam/steam_game_output.json"))
 
print(f"Nombre de records bruts : {df_raw.count()}")
df_raw.printSchema()

Nombre de records bruts : 55691
root
 |-- data: struct (nullable = true)
 |    |-- appid: long (nullable = true)
 |    |-- categories: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |    |-- ccu: long (nullable = true)
 |    |-- developer: string (nullable = true)
 |    |-- discount: string (nullable = true)
 |    |-- genre: string (nullable = true)
 |    |-- header_image: string (nullable = true)
 |    |-- initialprice: string (nullable = true)
 |    |-- languages: string (nullable = true)
 |    |-- name: string (nullable = true)
 |    |-- negative: long (nullable = true)
 |    |-- owners: string (nullable = true)
 |    |-- platforms: struct (nullable = true)
 |    |    |-- linux: boolean (nullable = true)
 |    |    |-- mac: boolean (nullable = true)
 |    |    |-- windows: boolean (nullable = true)
 |    |-- positive: long (nullable = true)
 |    |-- price: string (nullable = true)
 |    |-- publisher: string (nullable = true)
 |    |-- release_date: s

In [0]:
# Aperçu des premiers enregistrements
df_raw.select(
    col("id"),
    col("data").getField("name").alias("name"),
    col("data").getField("appid").alias("appid"),
    col("data").getField("type").alias("type")
).show(5, truncate=False)

+-------+---------------------------+-------+----+
|id     |name                       |appid  |type|
+-------+---------------------------+-------+----+
|10     |Counter-Strike             |10     |game|
|1000000|ASCENXION                  |1000000|game|
|1000010|Crown Trick                |1000010|game|
|1000030|Cook, Serve, Delicious! 3?!|1000030|game|
|1000040|细胞战争                   |1000040|game|
+-------+---------------------------+-------+----+
only showing top 5 rows


In [0]:
# Vérification des types disponibles
df_raw \
    .select(col("data").getField("type").alias("type")) \
    .groupBy("type") \
    .count() \
    .orderBy("count", ascending=False) \
    .show()

+--------+-----+
|    type|count|
+--------+-----+
|    game|55690|
|hardware|    1|
+--------+-----+



In [0]:
# On filtre sur type == "game" et on extrait tous les champs utiles
# via col("data").getField() pour naviguer dans la structure imbriquée

df_games = (
    df_raw 
        .filter(col("data").getField("type") == "game") 
        .select(
            col("data").getField("appid").alias("appid"),
            col("data").getField("name").alias("name"),
            col("data").getField("developer").alias("developer"),
            col("data").getField("publisher").alias("publisher"),
            col("data").getField("genre").alias("genre_raw"),
            col("data").getField("categories").alias("categories"),
            col("data").getField("owners").alias("owners_range"),
            col("data").getField("positive").cast(IntegerType()).alias("positive"),
            col("data").getField("negative").cast(IntegerType()).alias("negative"),
            col("data").getField("price").alias("price_raw"),
            col("data").getField("initialprice").alias("initialprice_raw"),
            col("data").getField("discount").alias("discount_raw"),
            col("data").getField("ccu").cast(IntegerType()).alias("ccu"),
            col("data").getField("languages").alias("languages_raw"),
            col("data").getField("release_date").alias("release_date_raw"),
            col("data").getField("required_age").alias("required_age_raw"),
            col("data").getField("platforms").getField("windows").alias("windows"),
            col("data").getField("platforms").getField("mac").alias("mac"),
            col("data").getField("platforms").getField("linux").alias("linux"),
            col("data").getField("header_image").alias("header_image")
        )
    )
 
print(f"Nombre de jeux : {df_games.count()}")
df_games.show(3, truncate=False)

Nombre de jeux : 55690
+-------+--------------+----------------------+------------------------+-------------------------------+-------------------------------------------------------------------------------------------------+------------------------+--------+--------+---------+----------------+------------+-----+-------------------------------------------------------------------------------------------------------------------------+----------------+----------------+-------+-----+-----+-----------------------------------------------------------------------------+
|appid  |name          |developer             |publisher               |genre_raw                      |categories                                                                                       |owners_range            |positive|negative|price_raw|initialprice_raw|discount_raw|ccu  |languages_raw                                                                                                            |release_date_raw|r

## **Nettoyage des colonnes**

In [0]:
df_games_clean = df_games

In [0]:
# cast prix en numérique
df_games_clean = (
    df_games_clean 
        .withColumn("price", col("price_raw").cast(FloatType())) 
        .withColumn("initialprice", col("initialprice_raw").cast(FloatType())) 
        .withColumn("discount", col("discount_raw").cast(FloatType()))
        .drop("price_raw", "initialprice_raw", "discount_raw")
)

df_games_clean.select("name", "price", "initialprice", "discount").show(10)

+------------------------------------+------+------------+--------+
|                                name| price|initialprice|discount|
+------------------------------------+------+------------+--------+
|                      Counter-Strike| 999.0|       999.0|     0.0|
|                           ASCENXION| 999.0|       999.0|     0.0|
|                         Crown Trick| 599.0|      1999.0|    70.0|
|                Cook, Serve, Deli...|1999.0|      1999.0|     0.0|
|                            细胞战争| 199.0|       199.0|     0.0|
|                             Zengeon| 799.0|      1999.0|    60.0|
|干支セトラ　陽ノ卷｜干支etc.　陽之卷|1299.0|      1299.0|     0.0|
|            Jumping Master(跳跳大咖)|   0.0|         0.0|     0.0|
|                       Cube Defender| 299.0|       299.0|     0.0|
|                Tower of Origin2-...|1399.0|      1399.0|     0.0|
+------------------------------------+------+------------+--------+
only showing top 10 rows


In [0]:
# Date de sortie — parsing + extraction année et mois
df_games_clean = ( 
    df_games_clean 
        .withColumn(
            "release_date", 
            coalesce(
                try_to_date(col("release_date_raw"), "yyyy/M/d"), 
                try_to_date(col("release_date_raw"), "yyyy/M"),   
                try_to_date(col("release_date_raw"), "yyyy")      
            )
        ) 
        .withColumn("release_year", year(col("release_date"))) 
        .withColumn("release_month", month(col("release_date")))
        .drop("release_date_raw")
)
 
# Vérification : combien de dates non parsées ?
n_null = df_games_clean.filter(col("release_date").isNull()).count()
print(f"Dates non parsées (null) : {n_null}")
 
df_games_clean.select("name", "release_date", "release_year", "release_month").show(10)


Dates non parsées (null) : 99
+------------------------------------+------------+------------+-------------+
|                                name|release_date|release_year|release_month|
+------------------------------------+------------+------------+-------------+
|                      Counter-Strike|  2000-11-01|        2000|           11|
|                           ASCENXION|  2021-05-14|        2021|            5|
|                         Crown Trick|  2020-10-16|        2020|           10|
|                Cook, Serve, Deli...|  2020-10-14|        2020|           10|
|                            细胞战争|  2019-03-30|        2019|            3|
|                             Zengeon|  2019-06-24|        2019|            6|
|干支セトラ　陽ノ卷｜干支etc.　陽之卷|  2019-01-24|        2019|            1|
|            Jumping Master(跳跳大咖)|  2019-04-08|        2019|            4|
|                       Cube Defender|  2019-01-06|        2019|            1|
|                Tower of Origin2-...|  2021-0

In [0]:
df_games_clean = (
    df_games_clean
        .withColumn("required_age",
            regexp_extract(col("required_age_raw"), r"(\d+)", 1).cast(IntegerType())
        )
        .drop("required_age_raw")
)

df_games_clean.groupBy("required_age") \
    .count() \
    .orderBy("required_age") \
    .show()

+------------+-----+
|required_age|count|
+------------+-----+
|           0|55029|
|           3|    3|
|           5|    1|
|           6|    4|
|           7|    3|
|           8|    3|
|           9|    1|
|          10|    7|
|          12|   32|
|          13|   26|
|          14|   10|
|          15|  265|
|          16|   38|
|          17|   38|
|          18|  223|
|          20|    1|
|          21|    1|
|          35|    1|
|         180|    4|
+------------+-----+



In [0]:
# wners_range — extraction de la borne basse numérique
df_games_clean = (
    df_games_clean 
        .withColumn("owners_lower",
            regexp_replace(
                regexp_extract(col("owners_range"), r"^([\d,]+)", 1),
                ",", ""
            ).cast(LongType())
        )
)
 
df_games_clean.select("owners_range", "owners_lower") \
    .distinct() \
    .orderBy("owners_lower") \
    .show(10, truncate=False)


+------------------------+------------+
|owners_range            |owners_lower|
+------------------------+------------+
|0 .. 20,000             |0           |
|20,000 .. 50,000        |20000       |
|50,000 .. 100,000       |50000       |
|100,000 .. 200,000      |100000      |
|200,000 .. 500,000      |200000      |
|500,000 .. 1,000,000    |500000      |
|1,000,000 .. 2,000,000  |1000000     |
|2,000,000 .. 5,000,000  |2000000     |
|5,000,000 .. 10,000,000 |5000000     |
|10,000,000 .. 20,000,000|10000000    |
+------------------------+------------+
only showing top 10 rows


In [0]:
# Reviews — calcul du total et du ratio positif
df_games_clean = (df_games_clean 
    .withColumn("total_reviews", col("positive") + col("negative")) 
    .withColumn("positive_ratio",
        when(
            col("total_reviews") > 0,
            col("positive").cast(DoubleType()) / col("total_reviews").cast(DoubleType())
        ).otherwise(
            lit(None).cast(DoubleType())
        )
    )
)
 
df_games_clean.select("name", "positive", "negative", "total_reviews", "positive_ratio") \
    .orderBy("total_reviews", ascending=False) \
    .show(10)


+--------------------+--------+--------+-------------+------------------+
|                name|positive|negative|total_reviews|    positive_ratio|
+--------------------+--------+--------+-------------+------------------+
|Counter-Strike: G...| 5943345|  787093|      6730438|0.8830547135268165|
| PUBG: BATTLEGROUNDS| 1185361|  908515|      2093876|0.5661084992616564|
|              Dota 2| 1534895|  317916|      1852811| 0.828414231133127|
|  Grand Theft Auto V| 1229265|  213379|      1442644| 0.852091714934523|
|Tom Clancy's Rain...|  942910|  143247|      1086157|0.8681157512219688|
|            Terraria| 1014711|   22380|      1037091| 0.978420408623737|
|     Team Fortress 2|  846407|   57423|       903830|0.9364670347299824|
|         Garry's Mod|  861240|   29998|       891238|0.9663412017889722|
|                Rust|  732513|  112154|       844667|0.8672210468740936|
|       Left 4 Dead 2|  643836|   16828|       660664|0.9745286560187932|
+--------------------+--------+-------

In [0]:
# Langues — split de la string en liste + comptage
df_games_clean = (df_games_clean 
    .withColumn("languages_list",
        split(
            regexp_replace(col("languages_raw"), r"\s*,\s*", ","),
            ","
        )
    ) \
    .withColumn("nb_languages",
        array_size(col("languages_list"))
    ) 
    .drop("languages_raw")
)
 
df_games_clean.select("name", "nb_languages", "languages_list") \
    .orderBy("nb_languages", ascending=False) \
    .show(10, truncate=False)

+----------------------------------------+------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|name                                    |nb_languages|languages_list                                                                                                                                                                                                                                                                                                                                                                                                                    |
+----------------------------------------+--------

In [0]:
# Genre — split de la string multi-valeurs en liste
df_games_clean = (df_games_clean 
    .withColumn("genres_list",
        split(
            regexp_replace(col("genre_raw"), r"\s*,\s*", ","),
            ","
        )
    ) \
    .drop("genre_raw")
)
 
df_games_clean.select("name", "genres_list").show(10, truncate=False)


+------------------------------------+----------------------------------------------------------------+
|name                                |genres_list                                                     |
+------------------------------------+----------------------------------------------------------------+
|Counter-Strike                      |[Action]                                                        |
|ASCENXION                           |[Action, Adventure, Indie]                                      |
|Crown Trick                         |[Adventure, Indie, RPG, Strategy]                               |
|Cook, Serve, Delicious! 3?!         |[Action, Indie, Simulation, Strategy]                           |
|细胞战争                            |[Action, Casual, Indie, Simulation]                             |
|Zengeon                             |[Action, Adventure, Indie, RPG]                                 |
|干支セトラ　陽ノ卷｜干支etc.　陽之卷|[Adventure, Indie, RPG, Strategy]             

## **DataFrame principal nettoyé — vérification finale**

In [0]:
df_clean = df_games_clean.select(
    "appid", "name", "developer", "publisher",
    "genres_list", "categories",
    "owners_range", "owners_lower",
    "positive", "negative", "total_reviews", "positive_ratio",
    "price", "initialprice", "discount",
    "ccu", "nb_languages", "languages_list",
    "release_date", "release_year", "release_month",
    "required_age", "windows", "mac", "linux",
    "header_image"
)
 
print(f"Shape finale : {df_clean.count()} lignes x {len(df_clean.columns)} colonnes")
df_clean.printSchema()

Shape finale : 55690 lignes x 26 colonnes
root
 |-- appid: long (nullable = true)
 |-- name: string (nullable = true)
 |-- developer: string (nullable = true)
 |-- publisher: string (nullable = true)
 |-- genres_list: array (nullable = true)
 |    |-- element: string (containsNull = false)
 |-- categories: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- owners_range: string (nullable = true)
 |-- owners_lower: long (nullable = true)
 |-- positive: integer (nullable = true)
 |-- negative: integer (nullable = true)
 |-- total_reviews: integer (nullable = true)
 |-- positive_ratio: double (nullable = true)
 |-- price: float (nullable = true)
 |-- initialprice: float (nullable = true)
 |-- discount: float (nullable = true)
 |-- ccu: integer (nullable = true)
 |-- nb_languages: integer (nullable = true)
 |-- languages_list: array (nullable = true)
 |    |-- element: string (containsNull = false)
 |-- release_date: date (nullable = true)
 |-- release_year: intege

In [0]:
# Statistiques descriptives sur les colonnes numériques
df_clean.select(
    "price", "discount",
    "positive", "negative", "total_reviews", "positive_ratio",
    "ccu", "nb_languages", "owners_lower"
).describe().show()

+-------+------------------+------------------+------------------+-----------------+------------------+-------------------+-----------------+------------------+-----------------+
|summary|             price|          discount|          positive|         negative|     total_reviews|     positive_ratio|              ccu|      nb_languages|     owners_lower|
+-------+------------------+------------------+------------------+-----------------+------------------+-------------------+-----------------+------------------+-----------------+
|  count|             55690|             55690|             55690|            55690|             55690|              55527|            55690|             55690|            55690|
|   mean| 773.2988687376549|2.6038247441192315|1470.7978093014904|241.8102352307416| 1712.608044532232| 0.7365350137972013|138.9621116897109|3.6326629556473335|78860.65720955288|
| stddev|1093.1394858517053|12.887191156585029| 30983.00621611397|5765.461884204928|35687.922958808886|0.

## **Création des DataFrames thématiques**

In [0]:
# df_genres — une ligne par genre via `explode()`
df_genres = (df_clean 
    .select(
        "appid", "name", "publisher", "developer",
        "positive_ratio", "total_reviews",
        "price", "owners_lower", "release_year",
        explode(col("genres_list")).alias("genre")
    ) 
    .withColumn("genre", trim(col("genre"))) \
    .filter(col("genre") != "")
)
 
print(f"df_genres : {df_genres.count()} lignes")
df_genres.show(10, truncate=False)

df_genres : 157110 lignes
+-------+---------------------------+------------------------+----------------------+------------------+-------------+------+------------+------------+---------+
|appid  |name                       |publisher               |developer             |positive_ratio    |total_reviews|price |owners_lower|release_year|genre    |
+-------+---------------------------+------------------------+----------------------+------------------+-------------+------+------------+------------+---------+
|10     |Counter-Strike             |Valve                   |Valve                 |0.9748127549487923|206414       |999.0 |10000000    |2000        |Action   |
|1000000|ASCENXION                  |PsychoFlux Entertainment|IndigoBlue Game Studio|0.84375           |32           |999.0 |0           |2021        |Action   |
|1000000|ASCENXION                  |PsychoFlux Entertainment|IndigoBlue Game Studio|0.84375           |32           |999.0 |0           |2021        |Adventure|
|1

In [0]:
# df_languages — une ligne par langue via `explode()`
df_languages = (df_clean 
    .select(
        "appid", "name",
        explode(col("languages_list")).alias("language")
    ) 
    .withColumn("language", trim(col("language"))) \
    .filter(col("language") != "")
)
 
print(f"df_languages : {df_languages.count()} lignes")
df_languages.show(10, truncate=False)

df_languages : 202293 lignes
+-------+--------------+-------------------+
|appid  |name          |language           |
+-------+--------------+-------------------+
|10     |Counter-Strike|English            |
|10     |Counter-Strike|French             |
|10     |Counter-Strike|German             |
|10     |Counter-Strike|Italian            |
|10     |Counter-Strike|Spanish - Spain    |
|10     |Counter-Strike|Simplified Chinese |
|10     |Counter-Strike|Traditional Chinese|
|10     |Counter-Strike|Korean             |
|1000000|ASCENXION     |English            |
|1000000|ASCENXION     |Korean             |
+-------+--------------+-------------------+
only showing top 10 rows


In [0]:
# df_categories — une ligne par catégorie via `explode()`
df_categories = (df_clean 
    .select(
        "appid", "name",
        explode(col("categories")).alias("category")
    ) 
    .withColumn("category", trim(col("category"))) \
    .filter(col("category") != "")
)
 
print(f"df_categories : {df_categories.count()} lignes")
df_categories.show(10, truncate=False)

df_categories : 191268 lignes
+-------+--------------+--------------------------+
|appid  |name          |category                  |
+-------+--------------+--------------------------+
|10     |Counter-Strike|Multi-player              |
|10     |Counter-Strike|Valve Anti-Cheat enabled  |
|10     |Counter-Strike|Online PvP                |
|10     |Counter-Strike|Shared/Split Screen PvP   |
|10     |Counter-Strike|PvP                       |
|1000000|ASCENXION     |Single-player             |
|1000000|ASCENXION     |Partial Controller Support|
|1000000|ASCENXION     |Steam Achievements        |
|1000000|ASCENXION     |Steam Cloud               |
|1000010|Crown Trick   |Single-player             |
+-------+--------------+--------------------------+
only showing top 10 rows


In [0]:
# df_platforms — format long Windows / Mac / Linux via `stack()`
df_platforms = (df_clean 
    .select(
        "appid", "name", "genres_list", "release_year",
        stack(
            lit(3),
            lit("Windows"), col("windows"),
            lit("Mac"),     col("mac"),
            lit("Linux"),   col("linux")
        ).alias("platform", "available")
    ) 
    .filter(col("available") == True)
)
 
print(f"df_platforms : {df_platforms.count()} lignes")
df_platforms.show(10)

df_platforms : 76901 lignes
+-------+------------------------------------+--------------------+------------+--------+---------+
|  appid|                                name|         genres_list|release_year|platform|available|
+-------+------------------------------------+--------------------+------------+--------+---------+
|     10|                      Counter-Strike|            [Action]|        2000| Windows|     true|
|1000000|                           ASCENXION|[Action, Adventur...|        2021| Windows|     true|
|1000010|                         Crown Trick|[Adventure, Indie...|        2020| Windows|     true|
|1000030|                Cook, Serve, Deli...|[Action, Indie, S...|        2020| Windows|     true|
|1000040|                            细胞战争|[Action, Casual, ...|        2019| Windows|     true|
|1000080|                             Zengeon|[Action, Adventur...|        2019| Windows|     true|
|1000100|干支セトラ　陽ノ卷｜干支etc.　陽之卷|[Adventure, Indie...|        2019| Windows|   

In [0]:
# df_publishers — agrégations par éditeur
df_publishers = (df_clean 
    .groupBy("publisher") 
    .agg(
        count("appid").alias("nb_games"),
        avg("price").alias("avg_price"),
        avg("positive_ratio").alias("avg_positive_ratio"),
        spark_sum("total_reviews").alias("total_reviews"),
        spark_max("release_year").alias("last_release_year")
    ) \
    .orderBy("nb_games", ascending=False)
)
 
print(f"df_publishers : {df_publishers.count()} publishers uniques")
df_publishers.show(20, truncate=False)

df_publishers : 29966 publishers uniques
+--------------------------+--------+------------------+------------------+-------------+-----------------+
|publisher                 |nb_games|avg_price         |avg_positive_ratio|total_reviews|last_release_year|
+--------------------------+--------+------------------+------------------+-------------+-----------------+
|Big Fish Games            |422     |949.7582938388625 |0.8391156965870649|7536         |2022             |
|8floor                    |202     |499.4950495049505 |0.7460713789979267|1485         |2022             |
|SEGA                      |165     |1434.9939393939394|0.855940198979354 |937419       |2022             |
|Strategy First            |151     |692.4370860927153 |0.5619221092131133|67844        |2022             |
|Square Enix               |141     |2082.1205673758864|0.7522290179435571|967072       |2022             |
|Choice of Games           |140     |534.6428571428571 |0.7397462956149702|7817         |2022  

## **Enregistrement des tables temporaires**

In [0]:
# df_clean.createOrReplaceTempView("steam_games")
# df_genres.createOrReplaceTempView("steam_genres")
# df_languages.createOrReplaceTempView("steam_languages")
# df_categories.createOrReplaceTempView("steam_categories")
# df_platforms.createOrReplaceTempView("steam_platforms")
# df_publishers.createOrReplaceTempView("steam_publishers")
 
# print("✅ Tables temporaires créées et prêtes pour le Notebook 02 :")
# for t in ["steam_games", "steam_genres", "steam_languages",
#           "steam_categories", "steam_platforms", "steam_publishers"]:
#     print(f"   • {t}")

# Au lieu de temp view, on va creer un delta table qui va persister entre les sessions de spark et reutiliser dans les autres notebooks 
df_clean.write.format("delta").mode("overwrite").saveAsTable("steam_games")
df_genres.write.format("delta").mode("overwrite").saveAsTable("steam_genres")
df_languages.write.format("delta").mode("overwrite").saveAsTable("steam_languages")
df_categories.write.format("delta").mode("overwrite").saveAsTable("steam_categories")
df_platforms.write.format("delta").mode("overwrite").saveAsTable("steam_platforms")
df_publishers.write.format("delta").mode("overwrite").saveAsTable("steam_publishers")

print("✅ Delta Tables sauvegardées et prêtes pour le Notebook 02 :")
for t in ["steam_games", "steam_genres", "steam_languages",
          "steam_categories", "steam_platforms", "steam_publishers"]:
    print(f"   • {t}")

✅ Delta Tables sauvegardées et prêtes pour le Notebook 02 :
   • steam_games
   • steam_genres
   • steam_languages
   • steam_categories
   • steam_platforms
   • steam_publishers
